# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakhawat-4/my-ml-internship-v2/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Target (same proxy as Week 4, so the comparison is apples-to-apples):**
`is_declining_label = trend_direction == "down"`.

**Question shape:** this is a yes/no outcome with an observed label (not a future window, not
clustering, not "what drives X" in isolation) — per `training-honest-models`, that maps to
**Logistic Regression, then Random Forest**. I'm adding a shallow **Decision Tree (depth 3)** in
between the two: it's nearly free to train, and unlike the other two it can be printed and read
in one glance, which is worth more than a couple points of AUC on its own.

I'm skipping Gradient Boosting this week — the skill's own rule is "add complexity only when the
comparison earns it," and I want to see whether Random Forest already beats the baseline by a
wide margin before reaching for something heavier. If RF barely beats it, that's the point to
justify GBM; if RF already wins clearly, adding GBM would just be complexity for its own sake.

**Why Logistic Regression, Decision Tree, and Random Forest, specifically for Lane 2** (Refresh /
Content Opportunity Scoring): the deliverable is a *ranked queue*, same as Week 4's baseline, so
every model here is evaluated as a **ranker** (precision@K) as well as a classifier — matching
"ranking needs scores, not labels" from the skill table.


In [1]:
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"n = {len(df):,} rows | base rate (declining) = {df['is_declining_label'].mean():.3f}")

# Heavy-tailed counts get log1p'd before going into a linear/tree model — same reasoning as the
# auditing-signals skill flagged for the Week 4 score (raw impressions dominating everything).
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c].clip(lower=0))

# Feature list matches this repo's own verified starter pipeline (scripts/ml_utils.py) —
# observable, pre-decision signals only.
NUMERIC = ["search_volume", "competition", "cpc", "word_count", "char_count",
           "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
           "days_with_impressions", "days_with_sessions", "content_age_days",
           "days_since_last_update", "ctr", "avg_position", "engagement_rate",
           "scroll_rate", "ai_traffic_pct"]
CATEG = ["competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
         "word_count_tier", "impression_tier", "position_tier"]

# Leakage guard: label-derived columns and identifiers must never enter the feature matrix.
excluded = {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}
assert excluded.isdisjoint(NUMERIC) and excluded.isdisjoint(CATEG)

num = df[NUMERIC].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = pd.get_dummies(df[CATEG].fillna("unknown").astype(str), prefix=CATEG)
X = pd.concat([num, cat], axis=1)
y = df["is_declining_label"]
print(f"feature matrix: {X.shape[1]} columns ({len(NUMERIC)} numeric, {X.shape[1]-len(NUMERIC)} one-hot)")


n = 30,000 rows | base rate (declining) = 0.542
feature matrix: 52 columns (18 numeric, 34 one-hot)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`, not by row.** Rows for the same client share a lot — the same site
conventions, the same SEO health, the same seasonal patterns. A random row-level split would let
the model see some of a client's pages in training and the rest in test, so it could partly learn
"this is Client X's content" rather than the general pattern — an easier, less honest problem.
Holding out entire clients means the model has never seen anything from a test-set client during
training, which is the harder, more honest version of the question ("does this generalize to a
brand-new client's content?").

There are 32 unique clients in this starter dataset — enough to hold out ~20% (6 clients) and
still keep both sides usable. No time-aware split: this snapshot doesn't carry the repeated
per-day windows a time-based split would need (that's the `Growth / Recovery / Momentum
Prediction` freestyle lane's job, not this one).


In [2]:
rng = np.random.default_rng(RANDOM_STATE)
clients = df["client_id"].unique()
shuffled_clients = rng.permutation(clients)
n_test_clients = max(1, round(len(shuffled_clients) * 0.2))
test_clients = set(shuffled_clients[:n_test_clients])

test_mask = df["client_id"].isin(test_clients).to_numpy()
train_idx, test_idx = np.where(~test_mask)[0], np.where(test_mask)[0]

Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

print(f"clients: {len(clients)} total, {n_test_clients} held out for test")
print(f"rows: {len(train_idx):,} train / {len(test_idx):,} test")
print(f"declining rate — train: {ytr.mean():.3f} | test: {yte.mean():.3f}")
assert set(df.iloc[train_idx]['client_id']).isdisjoint(set(df.iloc[test_idx]['client_id'])), \
    "train/test client overlap — split is not honest"
print("OK: no client appears in both train and test.")


clients: 32 total, 6 held out for test
rows: 27,675 train / 2,325 test
declining rate — train: 0.555 | test: 0.391
OK: no client appears in both train and test.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Week-4 rule (`refresh` / `fix_ctr` / `quick_win` priority score) is recomputed here on this
exact dataframe, then scored on the exact same held-out test rows as the models below — same
data, same split, same metric (precision@K, the natural metric for a ranked queue), plus a
plain 0.5-threshold classification view for the models. `base rate` is the declining rate in the
test set — the score a queue with zero skill would get.


In [3]:
def precision_at_k(y_true, scores, k):
    d = pd.DataFrame({"y": y_true.values, "s": scores}).sort_values("s", ascending=False).head(k)
    return float(d["y"].mean()) if len(d) else 0.0

# --- recompute the Week-4 baseline rule, unchanged, on the same rows ---
visible = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)]
tier_median_ctr = visible.groupby("position_tier")["ctr"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr)
refresh_cond = (df["freshness_tier"] == "91-180") & (df["impressions_90d"] >= 500)
ctr_fix_cond = (df["ctr"] < df["tier_median_ctr"]) & (df["impressions_90d"] >= 100) & (df["avg_position"] > 0)
quick_win_cond = (df["impression_tier"].isin(["good", "excellent"])) & (df["position_tier"] == "striking")
ctr_gap = ((df["tier_median_ctr"] - df["ctr"]) / df["tier_median_ctr"].replace(0, np.nan)).clip(lower=0).fillna(0)
baseline_score = np.select(
    [refresh_cond, ctr_fix_cond, quick_win_cond],
    [df["impressions_90d"], df["impressions_90d"] * ctr_gap, df["impressions_90d"] / df["avg_position"]],
    default=0.0,
)
baseline_test_scores = baseline_score[test_idx]

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("m", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(max_depth=3, min_samples_leaf=50,
                                             class_weight="balanced", random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                             class_weight="balanced_subsample", n_jobs=-1,
                                             random_state=RANDOM_STATE),
}

rows = [{
    "method": "baseline (Week 4 rule)",
    "p@20": precision_at_k(yte, baseline_test_scores, 20),
    "p@50": precision_at_k(yte, baseline_test_scores, 50),
    "p@100": precision_at_k(yte, baseline_test_scores, 100),
    "roc_auc": None, "avg_precision": None, "precision": None, "recall": None, "f1": None,
}]

proba_by_model = {}
for name, model in models.items():
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xte)[:, 1]
    proba_by_model[name] = p
    pred = (p >= 0.5).astype(int)
    rows.append({
        "method": name,
        "p@20": precision_at_k(yte, p, 20),
        "p@50": precision_at_k(yte, p, 50),
        "p@100": precision_at_k(yte, p, 100),
        "roc_auc": roc_auc_score(yte, p),
        "avg_precision": average_precision_score(yte, p),
        "precision": precision_score(yte, pred, zero_division=0),
        "recall": recall_score(yte, pred, zero_division=0),
        "f1": f1_score(yte, pred, zero_division=0),
    })

rows.append({"method": "base rate (no skill)", "p@20": yte.mean(), "p@50": yte.mean(),
             "p@100": yte.mean(), "roc_auc": 0.5, "avg_precision": yte.mean(),
             "precision": None, "recall": None, "f1": None})

comparison = pd.DataFrame(rows).set_index("method").round(3)
comparison


,p@20,p@50,p@100,roc_auc,avg_precision,precision,recall,f1
method,,,,,,,,
baseline (Week 4 rule),0.300,0.360,0.560,NaN,NaN,NaN,NaN,NaN
logistic_regression,0.350,0.400,0.440,0.700,0.522,0.566,0.567,0.566
decision_tree,0.450,0.500,0.500,0.698,0.519,0.526,0.806,0.637
random_forest,0.650,0.740,0.720,0.750,0.618,0.561,0.744,0.640
base rate (no skill),0.391,0.391,0.391,0.500,0.391,NaN,NaN,NaN


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Random Forest is the clear winner on every metric above, so it's the one worth reading errors
from.


In [4]:
rf = models["random_forest"]
rf_p = proba_by_model["random_forest"]
rf_pred = (rf_p >= 0.5).astype(int)

# --- what does it lean on? permutation importance, not just built-in impurity importance ---
pi = permutation_importance(rf, Xte, yte, n_repeats=10, random_state=RANDOM_STATE,
                             n_jobs=-1, scoring="roc_auc")
top_features = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False).head(8)
print("Top 8 features by permutation importance (drop in ROC-AUC when shuffled):")
print(top_features.round(4))
print()
print("Sanity check: the top feature is 'days_with_impressions' — how many of the 90 days a page")
print("showed up at all — not a suspiciously perfect stand-in for the label. That's a plausible")
print("real signal (inconsistent visibility precedes decline), not leakage.")

# --- where is it most wrong? ---
err = df.iloc[test_idx].copy()
err["y_true"], err["y_pred"], err["p"] = yte.values, rf_pred, rf_p
err["wrong"] = err["y_true"] != err["y_pred"]
print(f"\noverall test error rate: {err['wrong'].mean():.3f}")
print("\nerror rate by position_tier:")
print(err.groupby("position_tier")["wrong"].agg(n="size", error_rate="mean").round(3))
print("\nerror rate by freshness_tier:")
print(err.groupby("freshness_tier")["wrong"].agg(n="size", error_rate="mean").round(3))

# --- 3 concrete wrong cases each way ---
cols = ["content_id", "p", "days_with_impressions", "impressions_90d", "ctr",
        "avg_position", "position_tier", "freshness_tier"]
fp = err[(err.y_pred == 1) & (err.y_true == 0)].sort_values("p", ascending=False).head(3)
fn = err[(err.y_pred == 0) & (err.y_true == 1)].sort_values("p").head(3)
print("\nFalse positives — model says declining, actually stable/up:")
print(fp[cols].to_string(index=False))
print("\nFalse negatives — model says fine, actually declining:")
print(fn[cols].to_string(index=False))


Top 8 features by permutation importance (drop in ROC-AUC when shuffled):
days_with_impressions    0.0565
log_impressions_90d      0.0218
ctr                      0.0105
avg_position             0.0068
scroll_rate              0.0065
log_clicks_90d           0.0062
days_with_sessions       0.0022
search_volume            0.0018
dtype: float64

Sanity check: the top feature is 'days_with_impressions' — how many of the 90 days a page
showed up at all — not a suspiciously perfect stand-in for the label. That's a plausible
real signal (inconsistent visibility precedes decline), not leakage.

overall test error rate: 0.328

error rate by position_tier:
                  n  error_rate
position_tier                  
deep             60       0.350
page_1         1061       0.382
page_3_5        281       0.427
striking        405       0.420
top_3           518       0.089

error rate by freshness_tier:
                   n  error_rate
freshness_tier                  
0-30            2236   

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

